In [3]:
import requests
import json

base_url = "http://localhost:8000"

url = f"{base_url}/api/tools"

# make a get request to the url
response = requests.get(url)

# print the response
# make response pretty
print(json.dumps(response.json(), indent=4))

{
    "tools": [
        {
            "name": "get_current_time",
            "description": "Returns the current date and time in ISO 8601 format.",
            "parameters_schema": {
                "type": "object",
                "properties": {},
                "required": []
            }
        },
        {
            "name": "calculate",
            "description": "Evaluates a math expression. Supports +, -, *, /, //, %, **.",
            "parameters_schema": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression, e.g. '2 + 3 * 4'"
                    }
                },
                "required": [
                    "expression"
                ]
            }
        },
        {
            "name": "generate_uuid",
            "description": "Generates a random UUID v4. Useful when a unique identifier is needed.

In [4]:
url = f"{base_url}/health"

# make a get request to the url
response = requests.get(url)

# print the response
# make response pretty
print(json.dumps(response.json(), indent=4))

{
    "status": "ok",
    "available_providers": [
        "openai",
        "anthropic",
        "local_openai_compatible",
        "google_gemini"
    ]
}


In [ ]:
# Helpers
def pretty_print_response(resp):
    print(f"Status: {resp.status_code}")
    print("Body:")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(resp.text)

def pick_model(provider: str) -> str:
    models_resp = requests.get(f"{base_url}/api/providers/{provider}/models", timeout=30)
    models_resp.raise_for_status()
    models = models_resp.json().get("models", [])
    if not models:
        raise RuntimeError(f"No models available for provider '{provider}'")
    return models[0]

provider = "openai"
model = pick_model(provider)
print(f"Using provider={provider}, model={model}")

# --- POST /api/chat ---
chat_payload = {
    "provider": provider,
    "model": model,
    "messages": [
        {"role": "system", "content": "You are concise and return JSON when asked."},
        {"role": "user", "content": "Return a JSON object with keys: greeting and timestamp."},
    ],
    "temperature": 0.2,
    "max_tokens": 150,
}

chat_resp = requests.post(f"{base_url}/api/chat", json=chat_payload, timeout=90)
print("\n=== /api/chat request payload ===")
print(json.dumps(chat_payload, indent=2))
print("\n=== /api/chat response ===")
pretty_print_response(chat_resp)

# --- POST /api/chat/turn (first turn) ---
turn_payload_1 = {
    "provider": provider,
    "model": model,
    "message": "Give me three short facts about FastAPI.",
    "system_prompt": "You are a technical assistant. Be concise.",
    "temperature": 0.2,
    "max_tokens": 180,
}

turn_resp_1 = requests.post(f"{base_url}/api/chat/turn", json=turn_payload_1, timeout=90)
print("\n=== /api/chat/turn first request payload ===")
print(json.dumps(turn_payload_1, indent=2))
print("\n=== /api/chat/turn first response ===")
pretty_print_response(turn_resp_1)

turn_data_1 = turn_resp_1.json() if turn_resp_1.ok else {}
conversation_id = turn_data_1.get("conversation_id")

# --- POST /api/chat/turn (second turn) ---
if conversation_id:
    turn_payload_2 = {
        "conversation_id": conversation_id,
        "provider": provider,
        "model": model,
        "message": "Now summarize those facts in one sentence.",
        "temperature": 0.2,
        "max_tokens": 120,
    }
    turn_resp_2 = requests.post(f"{base_url}/api/chat/turn", json=turn_payload_2, timeout=90)
    print("\n=== /api/chat/turn second request payload ===")
    print(json.dumps(turn_payload_2, indent=2))
    print("\n=== /api/chat/turn second response ===")
    pretty_print_response(turn_resp_2)
else:
    print("\nSkipping second turn because no conversation_id was returned.")